# ZelusBench: Dimensionality

**Can the model maintain higher-dimensional state?**

This task tests representational load by running identical scenario structures
in 2D vs 3D space. The chain depth and other parameters are held constant —
only the number of coordinates changes.

Dimensions:
- **2D**: flat plane (x, y)
- **3D**: standard spatial (x, y, z)

In [ ]:
import kaggle_benchmarks as kbench
import pandas as pd
import numpy as np
import re
import os

os.environ["RENDER_SUBRUNS"] = "False"

In [ ]:
from zelusbench.scenarios.config import (
    ScenarioConfig, DAGStructure, QueryType,
    DistractorLevel, TransformDensity,
)
from zelusbench.scenarios.generator import ScenarioGenerator
from zelusbench.evaluation.parser import parse_model_response
from zelusbench.evaluation.scorer import score_query


def score_response(response_text, scenario):
    query_dicts = [q.to_dict() for q in scenario.queries]
    parts = re.split(r'\[Query\s+q_\d+\]', response_text)
    if len(parts) > 1:
        parts = parts[1:]
    if len(parts) != len(query_dicts):
        parts = [response_text] * len(query_dicts)
    return [score_query(qd, parse_model_response(rp, qd)) for qd, rp in zip(query_dicts, parts)]

In [ ]:
@kbench.task(store_task=False)
def dimensionality_scenario(llm, dim: int, seed: int) -> float:
    """Run one scenario in a given spatial dimension."""
    cfg = ScenarioConfig(
        dim=dim,
        min_chain_depth=4, max_chain_depth=6,
        dag_structure=DAGStructure.LINEAR,
        distractor_level=DistractorLevel.CLEAN,
        transform_density=TransformDensity.STATIC,
        query_types=[QueryType.POSITION, QueryType.DISTANCE],
        num_queries=3, num_splits=3,
        seed=seed,
    )
    gen = ScenarioGenerator(cfg)
    scenario = gen.generate(f"dim_{dim}d_s{seed}")

    response = llm.prompt(scenario.prompt)
    scores = score_response(response, scenario)
    avg = float(np.mean([s.score for s in scores]))

    for s in scores:
        kbench.assertions.assert_true(
            s.score > 0,
            expectation=(
                f"{s.query_id} ({dim}D): "
                f"model should handle {dim}D coordinates. Tier={s.tier.name}"
            )
        )
    return avg

In [ ]:
eval_data = pd.DataFrame([
    {"dim": d, "seed": seed}
    for d in [2, 3]
    for seed in range(10)
])

print(f"Evaluation matrix: {len(eval_data)} scenarios")
print(f"Dimensions: {sorted(eval_data.dim.unique())}")

In [ ]:
@kbench.task(name="zelusbench_dimensionality")
def zelusbench_dimensionality(llm) -> tuple[float, float]:
    """Dimensionality: accuracy by spatial dimension.

    Returns (mean_accuracy, std_dev).
    """
    with kbench.client.enable_cache():
        runs = dimensionality_scenario.evaluate(
            llm=[llm],
            evaluation_data=eval_data,
            n_jobs=2,
            timeout=180,
            remove_run_files=True,
            stop_condition=lambda r: len(r) == len(eval_data),
            max_attempts=60,
            retry_delay=10,
        )

    results_df = runs.as_dataframe()
    scores = results_df["result"].dropna().tolist()
    accuracy = float(np.mean(scores)) if scores else 0.0
    std = float(np.std(scores)) if scores else 0.0

    kbench.assertions.assert_true(
        len(scores) > 0,
        expectation="At least some scenarios should produce results"
    )
    return accuracy, std

In [ ]:
run = zelusbench_dimensionality.run(kbench.llm)
run

In [ ]:
%choose zelusbench_dimensionality